# Day 4: Chains - The Pipe

LangChain in 10 Days · KGP Talkie · Laxmi Kant Tiwari · kgptalkie.com

**Today:** join a template, a model and an unwrapper with one `|`. Chain model calls together, run many inputs at once, and stream the answer as it's written.

In [2]:
import time

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

model = ChatOpenRouter(model="nvidia/nemotron-3.5-lightning:free", temperature=0)

In [3]:
model

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openrouter': '0.2.7'}}, profile={'name': 'Laguna S 2.1 (free)', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': True, 'max_input_tokens': 262144, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': False, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x000001F067C90CE0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='poolside/laguna-s-2.1:free', temperature=0.0, model_kwargs={})

## 1. The two steps a chain replaces

In [5]:
summariser = ChatPromptTemplate.from_messages([
    ("system", "You summarise text for busy people."),
    ("user", "Summarise this in one sentence:\n\n{text}"),
])

article = """The city council approved a plan to plant 12,000 trees across
the eastern districts over the next four years. The scheme is funded by a
levy on new commercial developments, and the first 2,000 trees will go in
along the ring road where air quality readings have been worst."""

messages = summariser.invoke({"text": article})

model.invoke(messages).text

'The city council has approved a four-year plan to plant 12,000 trees in the eastern districts, funded by a levy on new commercial developments, with the first 2,000 trees being planted along the ring road where air quality is poorest.'

Two steps: build the messages, then send them. You will write that pair
thousands of times, so LangChain lets you join them once.

## 2. Join a template and a model with the pipe

In [6]:
chain = summariser | model

chain.invoke({"text": article}).text

'The city council has approved a four-year plan to plant 12,000 trees in eastern districts, funded by a commercial development levy, with the first 2,000 trees going along the ring road where air quality is poorest.'

`summariser | model` is one object now. Give it a dictionary, get a reply.

`|` is Python's or-operator, taught to hand output from the left to the right.

## 3. Add StrOutputParser to get a plain string

You usually want the plain string, not the reply object.

In [7]:
from langchain_core.output_parsers import StrOutputParser

chain = summariser | model | StrOutputParser()

chain.invoke({"text": article})

'The city council has approved a four-year plan to plant 12,000 trees in eastern districts, funded by a levy on new commercial developments, with the first 2,000 trees being planted along the ring road where air quality is poorest.'

No `.text` at the end, the parser did that.

**Read the chain left to right:** dictionary in, messages out of the template, a reply out of the model, a string out of the parser. Easier than reading nested calls from the inside out.

## 4. Look inside a chain: its steps and its inputs

A chain is not a black box. It is a list of the things you joined.

In [8]:
chain

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You summarise text for busy people.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='Summarise this in one sentence:\n\n{text}'), additional_kwargs={})])
| ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openrouter': '0.2.7'}}, profile={'name': 'Laguna S 2.1 (free)', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': True, 'max_input_tokens': 262144, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': Tru

In [9]:
[type(step).__name__ for step in chain.steps]

['ChatPromptTemplate', 'ChatOpenRouter', 'StrOutputParser']

The three objects, in order. It also knows what input it expects, worked out from the template.

In [10]:
list(chain.get_input_jsonschema()["properties"])

['text']

## 5. Every piece is a Runnable: invoke, batch, stream

The template, model, parser, and the whole chain share three verbs: `.invoke()`, `.batch()`, `.stream()`. That shared contract is the only reason `|` works, no type registry, just an agreement on methods.

In [11]:
summariser.invoke({"text": "hello"})

ChatPromptValue(messages=[SystemMessage(content='You summarise text for busy people.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Summarise this in one sentence:\n\nhello', additional_kwargs={}, response_metadata={})])

In [ ]:
model.invoke("Say hi in three words.").text

AIMessage(content='Hello, nice to meet you!', additional_kwargs={}, response_metadata={'model_name': 'poolside/laguna-s-2.1:free', 'id': 'gen-1786354578-p5mtB14YqOW2iCUrdhb1', 'created': 1786354578, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--019feb07-a9f2-7192-ab7b-5a0982348ec9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 8, 'total_tokens': 57, 'input_token_details': {'cache_read': 32, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})

In [13]:
from langchain.messages import AIMessage
StrOutputParser().invoke(AIMessage(content='Hello, nice to meet you!', additional_kwargs={}, response_metadata={'model_name': 'nvidia/nemotron-3.5-lightning:free', 'id': 'gen-1786354578-p5mtB14YqOW2iCUrdhb1', 'created': 1786354578, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--019feb07-a9f2-7192-ab7b-5a0982348ec9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 8, 'total_tokens': 57, 'input_token_details': {'cache_read': 32, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}))


'Hello, nice to meet you!'

In [14]:
StrOutputParser().invoke(model.invoke("Say hi in three words."))

'Hello there, friend!'

## 6. Chain two model calls together

Summarise the article, then turn the summary into a headline. The second template needs a dictionary with a `summary` key; the first chain produces a plain string, so something has to wrap it.

In [15]:
headline_writer = ChatPromptTemplate.from_messages([
    ("system", "You write short newspaper headlines. Five words at most."),
    ("user", "Write a headline for this summary:\n\n{summary}"),
])

In [16]:
time.sleep(4)

two_step = (
    chain
    | (lambda summary: {"summary": summary})
    | headline_writer
    | model
    | StrOutputParser()
)

two_step.invoke({"text": article})

'City Council Approves 12,000 Tree Initiative'

A plain Python function dropped into the pipe, LangChain wraps it for you. The trick to composing these: at each `|`, check what the left produces and what the right needs, and bridge the gap with a small function.

### Two chains in a plain Python function

Often you want the summary **and** the headline, not just the last thing.

In [17]:
time.sleep(4)

def summarise_and_title(inputs):
    summary = chain.invoke(inputs)
    headline = (headline_writer | model | StrOutputParser()).invoke({"summary": summary})
    return {"summary": summary, "headline": headline}

summarise_and_title({"text": article})

{'summary': 'The city council has approved a four-year plan to plant 12,000 trees in eastern districts, funded by a commercial development levy, with the first 2,000 trees going along the ring road where air quality is poorest.',
 'headline': 'City Council Approves 12,000-Tree Plan'}

Plain Python, calling two chains. Not everything has to be one long pipe, a readable function beats a clever expression.

## 7. Run many inputs at once with .batch()

`.batch()` sends several inputs together rather than one after another.

In [19]:
time.sleep(6)

articles = [
    {"text": "Rainfall in the northern hills was 40 percent above average in July."},
    {"text": "The new bridge cut the average commute from 55 minutes to 32."},
    {"text": "Library visits rose for the first time in nine years."},
]

chain.batch(articles)

['July saw rainfall in the northern hills 40% above average.',
 'The new bridge reduced the average commute time from 55 minutes to 32 minutes.',
 'Library visits have increased for the first time in nine years.']

In [18]:
chain

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You summarise text for busy people.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='Summarise this in one sentence:\n\n{text}'), additional_kwargs={})])
| ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openrouter': '0.2.7'}}, profile={'name': 'Laguna S 2.1 (free)', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': True, 'max_input_tokens': 262144, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': Tru

Three summaries in roughly the time of one.

**Careful on a free key:** `.batch()` fires calls concurrently, which trips the per-minute limit. You can hold it back.

In [ ]:
time.sleep(6)

chain.batch(articles, config={"max_concurrency": 1})

`max_concurrency: 1` runs them one at a time. Slower, much less likely to fail when twenty people share the same limit.

### Keep the good results with return_exceptions

By default, one failing input raises the whole `.batch()` call, losing the good results too.

In [23]:
mixed = [
    {"text": "The new bridge cut the average commute from 55 minutes to 32."},
    {"wrong_key": "this input is missing the text variable"},
]

results = chain.batch(mixed, return_exceptions=True)

results

['The new bridge reduced the average commute time by 23 minutes, from 55 to 32 minutes.',
 KeyError("Input to ChatPromptTemplate is missing variables {'text'}.  Expected: ['text'] Received: ['wrong_key']\nNote: if you intended {text} to be part of the string and not a variable, please escape it with double curly braces like: '{{text}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT ")]

In [22]:
time.sleep(6)

mixed = [
    {"text": "The new bridge cut the average commute from 55 minutes to 32."},
    {"wrong_key": "this input is missing the text variable"},
]

results = chain.batch(mixed, return_exceptions=True)

for r in results:
    if isinstance(r, Exception):
        print("failed: " + type(r).__name__)
    else:
        print("ok: " + r)

ok: The new bridge reduced the average commute time from 55 minutes to 32 minutes.
failed: KeyError


`return_exceptions=True` puts the error in the list instead of raising it, test which is which with `isinstance`.

For real data this is usually what you want: process what you can, collect failures for later.

## 8. Stream the answer with .stream()

`.stream()` hands you the answer in pieces as it arrives.

In [24]:
chain.stream({"text": article})

<generator object RunnableSequence.stream at 0x000001F01C8097B0>

In [26]:
# time.sleep(6)

for piece in chain.stream({"text": article}):
    print(piece, end="")

The city council has approved planting 12,000 trees across eastern districts over four years, funded by a commercial development levy, with the first 2,000 trees going along the ring road where air quality is worst.

Not faster, the last word arrives at the same moment either way. What changes: the user stops staring at a blank screen, why every chatbot does this.

It streams because **every** step supports streaming. A step that must see the whole input first, like the wrapping function in section 6, quietly kills streaming before that point.

## 9. Add a backup model with .with_fallbacks()

Free models fail. `.with_fallbacks()` lets you name a second one to try.

In [27]:
model

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openrouter': '0.2.7'}}, profile={'name': 'Laguna S 2.1 (free)', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': True, 'max_input_tokens': 262144, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': False, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x000001F067C90CE0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='poolside/laguna-s-2.1:free', temperature=0.0, model_kwargs={})

In [28]:
backup = ChatOpenRouter(model="meta-llama/llama-3.3-70b-instruct:free", temperature=0)

resilient = model.with_fallbacks([backup])

safe_chain = summariser | resilient | StrOutputParser()

In [29]:
# time.sleep(6)

safe_chain.invoke({"text": article})

'The city council has approved a four-year plan to plant 12,000 trees in eastern districts, funded by a levy on new commercial developments, with the first 2,000 trees going along the ring road where air quality is worst.'

If the first model raises, the second is tried with the same input, the chain around it doesn't change. Worth knowing on a free tier, where a 429 is a matter of when, not if.